# Polish benchmark: OCR engines and vision-language models

Generates the synthetic Polish dataset (TRDG, printed and handwritten fonts, *Lalka*) and the Gold Set (PHSF invocation
scans), runs nine models on both, and scores them. Each model runs in its own environment.

Models: Qwen2.5-VL 7B, InternVL3-8B, DeepSeek-VL2-tiny, Phi-3.5-vision, Pixtral-12B, GOT-OCR2, PaddleOCR, Tesseract,
EasyOCR.

**Runtime → Change runtime type:** A100. L4 (22 GB) skips Pixtral, T4 (15 GB) also Qwen and InternVL.

**Before running:** `research/make_bundle.sh`, then put `research/ocr_bundle.zip` in `MyDrive/ocr-benchmark/`. Results go to
`MyDrive/ocr-benchmark/research/`. After a disconnect use Run all again: the datasets are regenerated from the same seed,
finished models are skipped and an interrupted one continues.

**Size:** `N` images per synthetic cell (22 cells) and `GOLD_N` scans of the 1968 (`None` = all). Everything must fit in about
three hours on an A100, so `GOLD_N` is a random sample.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

from pathlib import Path
from google.colab import drive

N = 60
GOLD_N = 300
SEED = 42

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/ocr-benchmark")
OUT = DRIVE / "research"
OUT.mkdir(parents=True, exist_ok=True)
print("results:", OUT)

In [ ]:
import shutil, zipfile

bundle = next((p for p in (DRIVE / "ocr_bundle.zip", Path("/content/ocr_bundle.zip")) if p.exists()), None)
assert bundle, "ocr_bundle.zip not found in MyDrive/ocr-benchmark or /content"
shutil.rmtree("/content/ocr", ignore_errors=True)
zipfile.ZipFile(bundle).extractall("/content/ocr")
%cd /content/ocr
!pip install -q uv jiwer editdistance scipy
!apt-get -qq install -y tesseract-ocr tesseract-ocr-pol > /dev/null
!ls research/gold/images/*.png | wc -l

In [ ]:
!uv venv -q --python 3.11 /content/gen
!uv pip install -q --python /content/gen/bin/python pillow numpy fonttools opencv-python-headless tqdm requests arabic-reshaper python-bidi diffimg wikipedia
!uv pip install -q --python /content/gen/bin/python --no-deps trdg==1.8.0
!/content/gen/bin/python research/generate_dataset.py --n {N} --seed {SEED}
!python research/prepare_gold.py --n {GOLD_N} --seed {SEED}

In [ ]:
!python common/run_models.py --out "{OUT}" --language Polish \
    --models got paddle tesseract easyocr deepseek qwen7b internvl pixtral phi \
    --dataset synthetic research/data labels.csv --dataset gold research/gold/data labels.csv \
    --option temperature=0 --option max_new_tokens=512

In [ ]:
import json, pandas as pd

!python research/analyze.py --results "{OUT}"
runs = json.loads((OUT / "runs.json").read_text())
pd.DataFrame({m: {"status": r["status"], **{f"{d}_s/img": v["sec_per_image"] for d, v in r.get("datasets", {}).items()},
                  "total_min": round(r.get("total_s", 0) / 60, 1)} for m, r in runs.items()}).T